## Tahap Pertama: Data Understanding

In [1]:
import pandas as pd

df = pd.read_csv("rawdata.csv")
df.head()

,Nama,Rating,Jumlah Ulasan,Website,NoTelp,Alamat,Status
0,PT INKA (Persero) Banyuwangi,"4,7",127,https://www.inka.co.id/,8113476740,"Jalan INKA Banyuwangi, Lkr. Kp. Baru, Bulusan,...",Cold Prospek
1,PT. Pelindo Properti Indonesia Boom Marina Ban...,"4,5",250,NaN,333423282,"Jl. Ikan Cucut No.27, Kampungmandar, Kec. Bany...",Cold Prospek
2,Sun Osing Beach,"4,5",2200,NaN,82216211211,"Jl. Situbondo - Banyuwangi No.112, Lkr. Kp. Ba...",Cold Prospek
3,Plutos Billiards,"4,9",29,NaN,8121776869,"Jl. Kh. Hasyim Asyari, Dusun Krajan, Genteng W...",Hot Prospek
4,Alocasia Cafe and Eatery,"4,5",246,NaN,NaN,"Jl. Riau Gg. Paving, Krajan Barat, Sumbersari,...",Cold Prospek


In [2]:
df['Id'] = range(1, len(df) + 1)
df = df[['Id', 'Nama', 'Rating', 'Jumlah Ulasan', 'Website', 'NoTelp', 'Alamat', 'Status']]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Id             145 non-null    int64 
 1   Nama           145 non-null    object
 2   Rating         145 non-null    object
 3   Jumlah Ulasan  145 non-null    int64 
 4   Website        38 non-null     object
 5   NoTelp         118 non-null    object
 6   Alamat         141 non-null    object
 7   Status         145 non-null    object
dtypes: int64(2), object(6)
memory usage: 9.2+ KB


In [3]:
df.isnull().sum()

Id                 0
Nama               0
Rating             0
Jumlah Ulasan      0
Website          107
NoTelp            27
Alamat             4
Status             0
dtype: int64

## Tahap Kedua: Data Preparation

In [4]:
df = df.dropna(subset=['NoTelp'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 118 entries, 0 to 144
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Id             118 non-null    int64 
 1   Nama           118 non-null    object
 2   Rating         118 non-null    object
 3   Jumlah Ulasan  118 non-null    int64 
 4   Website        37 non-null     object
 5   NoTelp         118 non-null    object
 6   Alamat         114 non-null    object
 7   Status         118 non-null    object
dtypes: int64(2), object(6)
memory usage: 8.3+ KB


In [5]:
df_model = df.copy()
df_model = df_model[['Id', 'Rating', 'Jumlah Ulasan', 'Website', 'Status']]

df_model.head()

,Id,Rating,Jumlah Ulasan,Website,Status
0,1,"4,7",127,https://www.inka.co.id/,Cold Prospek
1,2,"4,5",250,NaN,Cold Prospek
2,3,"4,5",2200,NaN,Cold Prospek
3,4,"4,9",29,NaN,Hot Prospek
5,6,"4,6",426,NaN,Cold Prospek


In [6]:
df_model['Rating'] = df_model['Rating'].astype(str).str.replace(',', '.', regex=False)
df_model['Jumlah Ulasan'] = df_model['Jumlah Ulasan'].astype(str).str.replace('.', '', regex=False)
df_model['Website'] = df_model['Website'].fillna('')
df_model['Website'] = df_model['Website'].apply(lambda x: 1 if x != '' else 0)
df_model['Status'] = df_model['Status'].apply(lambda x: 1 if x != 'Cold Prospek' else 0)

df_model.head(5)

,Id,Rating,Jumlah Ulasan,Website,Status
0,1,4.7,127,1,0
1,2,4.5,250,0,0
2,3,4.5,2200,0,0
3,4,4.9,29,0,1
5,6,4.6,426,0,0


In [7]:
df_model['Rating'] = pd.to_numeric(df_model['Rating'], errors='coerce')
df_model['Jumlah Ulasan'] = pd.to_numeric(df_model['Jumlah Ulasan'], errors='coerce')
df_model = df_model.dropna(subset=['Rating', 'Jumlah Ulasan'])

df_model.info()

<class 'pandas.core.frame.DataFrame'>
Index: 118 entries, 0 to 144
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             118 non-null    int64  
 1   Rating         118 non-null    float64
 2   Jumlah Ulasan  118 non-null    int64  
 3   Website        118 non-null    int64  
 4   Status         118 non-null    int64  
dtypes: float64(1), int64(4)
memory usage: 5.5 KB


In [8]:
from sklearn.model_selection import train_test_split

x = df_model[['Rating', 'Jumlah Ulasan', 'Website']]
y = df_model['Status']

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
)

print("Jumlah data keseluruhan:", len(df_model))
print("Data training:", len(x_train))
print("Data testing:", len(x_test))

Jumlah data keseluruhan: 118
Data training: 94
Data testing: 24


## Tahap Ketiga: Modeling Random Forest

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    random_state=42
)

rf_model.fit(x_train, y_train)

y_prob = rf_model.predict_proba(x_test)
y_prob_df = pd.DataFrame(y_prob, columns=['Prob_ColdProspek', 'Prob_HotProspek'])
print(y_prob_df.head()) 

   Prob_ColdProspek  Prob_HotProspek
0              0.08             0.92
1              0.46             0.54
2              0.02             0.98
3              0.01             0.99
4              0.19             0.81


In [10]:
y_pred = rf_model.predict(x_test)
y_hasil = x_test.copy()
y_hasil['Prediksi_Status'] = y_pred
print(y_hasil.head())

     Rating  Jumlah Ulasan  Website  Prediksi_Status
71      4.9             25        0                1
110     4.9            373        1                1
5       4.6            426        0                1
60      4.9            350        0                1
31      4.1            698        0                1


In [11]:
feature_importance = pd.DataFrame({
    'Fitur': x.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(feature_importance)

           Fitur  Importance
1  Jumlah Ulasan    0.688556
0         Rating    0.283018
2        Website    0.028426


## Evaluasi Model

In [12]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[ 4  7]
 [ 2 11]]


In [13]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.625


In [14]:
precision = precision_score(y_test, y_pred)
print("Precision:", precision)

Precision: 0.6111111111111112


In [15]:
recall = recall_score(y_test, y_pred)
print("Recall:", recall)

Recall: 0.8461538461538461


## Tahap Keempat: Simpan Model ke model.pkl

In [16]:
import pickle
import os

MODEL_PATH = os.path.join(os.getcwd(), "model.pkl")

with open(MODEL_PATH, "wb") as f:
    pickle.dump(rf_model, f)

print(f"Model berhasil disimpan ke: {MODEL_PATH}")
print(f"Ukuran file : {os.path.getsize(MODEL_PATH):,} bytes")

Model berhasil disimpan ke: /home/enjuy/Documents/Ngoding/Python/Python---Random-Forest/EDA/model.pkl
Ukuran file : 438,318 bytes


## Tahap Kelima: Cek model.pkl

In [17]:
with open(MODEL_PATH, "rb") as f:
    loaded_model = pickle.load(f)

y_pred_verify = loaded_model.predict(x_test)

match = (y_pred_verify == y_pred).all()
print(f"Verifikasi prediksi identik dengan model asli: {match}")
print(f"Jumlah estimator : {loaded_model.n_estimators}")
print(f"Fitur yang digunakan : {loaded_model.feature_names_in_.tolist()}")

Verifikasi prediksi identik dengan model asli: True
Jumlah estimator : 100
Fitur yang digunakan : ['Rating', 'Jumlah Ulasan', 'Website']
